# Telco Customer Churn - Data Cleaning

#### The goal of this notebook is to clean and preprocess the raw dataset by handling missing values, correcting data types and removing unnecessary or inconsistent features, so it can be used for further analysis and modeling.

#### Note:
##### The raw dataset is not included in this repository.

In [30]:
# Import required libraries
import pandas as pd
import numpy as np

In [31]:
# Display all columns
pd.set_option('display.max_columns',None)

### 1. Load Dataset 

In [32]:
df = pd.read_excel("../data/raw/Telco_customer_churn.xlsx")
df.head(3)

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved


### 2. Remove Redundant and Leakage Columns

The following columns are removed as they either act as identifiers, duplicate the target variable, or contain post-outcome information that can lead to data leakage.

In [33]:
drop_cols = ['CustomerID','Churn Label','Churn Score', 'CLTV', 'Churn Reason']
df = df.drop(columns=drop_cols)

df.head(3)

,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value
0,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
1,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
2,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,1


### 3 : Convert columns with numeric meaning to proper numeric type

In [34]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

In [35]:
df['Total Charges'].dtype

dtype('float64')

### 4 : Check missing values after data type correction

In [36]:
df.isnull().sum()

Count                 0
Country               0
State                 0
City                  0
Zip Code              0
Lat Long              0
Latitude              0
Longitude             0
Gender                0
Senior Citizen        0
Partner               0
Dependents            0
Tenure Months         0
Phone Service         0
Multiple Lines        0
Internet Service      0
Online Security       0
Online Backup         0
Device Protection     0
Tech Support          0
Streaming TV          0
Streaming Movies      0
Contract              0
Paperless Billing     0
Payment Method        0
Monthly Charges       0
Total Charges        11
Churn Value           0
dtype: int64

In [37]:
df['Total Charges'].isnull().sum()

11

There are 11 instances with null value in **"Total Charges"** column. 

In [38]:
df[df['Total Charges'].isnull()].head(3)

,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value
2234,1,United States,California,San Bernardino,92408,"34.084909, -117.258107",34.084909,-117.258107,Female,No,Yes,No,0,No,No phone service,DSL,Yes,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,0
2438,1,United States,California,Independence,93526,"36.869584, -118.189241",36.869584,-118.189241,Male,No,No,No,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,0
2568,1,United States,California,San Mateo,94401,"37.590421, -122.306467",37.590421,-122.306467,Female,No,Yes,No,0,Yes,No,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,0


The total charges is given by:
       **Total charges = Monthly charges x Tenure**
       
Since these customers have a tenure of 0, their total charges are expected to be 0. Their missing values can be safely replaced with 0.

In [39]:
# fill the missing values with 0
df['Total Charges'] = df['Total Charges'].fillna(0)

In [40]:
df['Total Charges'].isnull().sum()

0

### 5: Check for Duplicates

In [41]:
df.duplicated().sum()

0

In [42]:
print('Shape after cleaning:', df.shape)

Shape after cleaning: (7043, 28)


### 6: Save Cleaned Dataset

In [43]:
# Save cleaned dataset for downstream tasks (EDA, feature engineering, modeling)
df.to_csv("../data/processed/telco_churn_cleaned.csv", index=False)

### Summary of Cleaning Steps:

- Removed columns:
   - **CustomerID** → unique identifier, not useful for prediction
   - **Churn Label** → duplicate of target varibale
   - **Churn Score, CLTV, Churn Reason** → derived after churn, leading to data leakage
   
- Converted **'Total Charges'** to numeric data type.
- Identified missing values in **'Total Charges'** and replaced with 0, as these correspond to customers with zero tenure.
- Verfied that there are no remaining missing values or duplicate records, ensuring data quality for modeling.